# DSCI 552 — Homework 3

Name: Zhicheng Zhang  
GitHub Username: Hustzz-Max  
USC ID: 8006795737

## 1(a)

Dataset source: [UCI AReM](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion).

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/AReM/AReM")
csv_files = sorted(data_dir.glob("*/dataset*.csv"))
print("Number of instances:", len(csv_files))

columns = [
    "time",
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23",
]

sample = pd.read_csv(
    data_dir / "bending1" / "dataset1.csv",
    comment="#",
    header=None,
    names=columns,
)
print("Sample shape:", sample.shape)
display(sample.head())

Number of instances: 88
Sample shape: (480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


## 1(b)

In [2]:
records = []
for path in csv_files:
    activity = path.parent.name
    dataset_id = int(path.stem.replace("dataset", ""))
    test_limit = 2 if activity in ["bending1", "bending2"] else 3
    split = "test" if dataset_id <= test_limit else "train"

    records.append({
        "activity": activity,
        "dataset_id": dataset_id,
        "path": path,
        "split": split,
    })

instance_info = pd.DataFrame(records)

train_files = instance_info.loc[
    instance_info["split"] == "train", "path"
].tolist()

test_files = instance_info.loc[
    instance_info["split"] == "test", "path"
].tolist()

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))
display(
    pd.crosstab(instance_info["activity"], instance_info["split"])
    .reindex(columns=["train", "test"])
)

Training instances: 69
Test instances: 19


split,train,test
activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3


## 1(c)

### 1(c)(i)
Common time-domain features:
Mean and median,
Minimum and maximum,
Standard deviation and variance,
Quartiles and interquartile range,
Skewness and kurtosis,
Energy and root mean square,
Autocorrelation and mean absolute change.

Reference: [tsfresh feature documentation](https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html).

### 1(c)(ii)

In [3]:
def read_arem(path):
    rows = []

    for line_number, line in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(),
        start=1,
    ):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        values = line.rstrip(",").replace(",", " ").split()
        if len(values) != 7:
            raise ValueError(
                f"{path}, line {line_number}: expected 7 values"
            )
        rows.append([float(value) for value in values])
    return pd.DataFrame(rows, columns=columns)

recordings = {
    path: read_arem(path)
    for path in instance_info["path"]
}

record_lengths = pd.Series({
    str(path.relative_to(data_dir)): len(df)
    for path, df in recordings.items()
}, name="observations")

print("Number of instances:", len(recordings))
print("Files with an observation count other than 480:")
display(record_lengths[record_lengths != 480])

Number of instances: 88
Files with an observation count other than 480:


sitting\dataset8.csv    479
Name: observations, dtype: int64

In [4]:
feature_rows = []

for path in instance_info["path"]:
    df = recordings[path]
    row = {}

    for i, column in enumerate(columns[1:], start=1):
        series = df[column]
        statistics = {
            "min": series.min(),
            "max": series.max(),
            "mean": series.mean(),
            "median": series.median(),
            "std": series.std(ddof=1),
            "q1": series.quantile(0.25),
            "q3": series.quantile(0.75),
        }
        for name, value in statistics.items():
            row[f"{name}_{i}"] = value
    feature_rows.append(row)

features = pd.DataFrame(feature_rows)
metadata = instance_info[
    ["activity", "dataset_id", "split"]
].reset_index(drop=True)

feature_table = pd.concat([metadata, features], axis=1)
feature_table.index = range(1, len(feature_table) + 1)
feature_table.index.name = "Instance"

print("Feature matrix shape:", features.shape)
print("Missing feature values:", features.isna().sum().sum())
display(feature_table.head())

Feature matrix shape: (88, 42)
Missing feature values: 0


,activity,dataset_id,split,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,...,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,
1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


### 1(c)(iii)

In [5]:
import numpy as np

rng = np.random.default_rng(552)
n_bootstrap = 10000
X = features.to_numpy(dtype=float)
n_instances, n_features = X.shape

sample_indices = rng.integers(
    0,
    n_instances,
    size=(n_bootstrap, n_instances),
)
bootstrap_stds = np.empty((n_bootstrap, n_features))

for j in range(n_features):
    bootstrap_stds[:, j] = X[sample_indices, j].std(
        axis=1,
        ddof=1,
    )

lower, upper = np.quantile(
    bootstrap_stds,
    [0.05, 0.95],
    axis=0,
)

std_results = pd.DataFrame({
    "std_estimate": features.std(ddof=1),
    "ci90_lower": lower,
    "ci90_upper": upper,
})
std_results.index.name = "feature"

print("Result shape:", std_results.shape)
display(std_results.round(4))

Result shape: (42, 3)


,std_estimate,ci90_lower,ci90_upper
feature,,,
min_1,9.5700,8.2569,10.7417
max_1,4.3944,3.3675,5.2850
mean_1,5.3357,4.7022,5.8823
median_1,5.4401,4.7905,5.9924
std_1,1.7722,1.5688,1.9480
q1_1,6.1536,5.5739,6.6362
q3_1,5.1389,4.3417,5.8503
min_2,0.0000,0.0000,0.0000
max_2,5.0627,4.6126,5.4077


### 1(c)(iv)
I'd like to select mean, standard deviation, and max. Mean shows the average level of the signal, which can be used to distinguish different activities. Standard deviation measures fluctuations, which can help distinguish activities with different movement patterns. Max records the highest signal value, which can provide extra information.

## 3.7.4

### (a)
The cubic model will have training RSS less than or equal to the linear model. Because the linear model is a special case of the cubic model, when the coefficients of $X^2$ and $X^3$ are zero. If the data contains noise, the cubic model can provide a better fit.

### (b)
We generally expect the linear model to have lower test RSS. Because the true relationship is linear, the extra terms in the cubic model may fit noise and lead to overfitting. This is an expectation, not a guarantee for every test set.

### (c)
The cubic model will have training RSS less than or equal to the linear model. A more flexible model can always fit the training data at least as well as a simpler nested model.

### (d)
There is not enough information to tell. If the true relationship is strongly nonlinear, the cubic model may have lower test RSS. If it is only slightly nonlinear, the linear model may perform better because it has lower variance.